# 01 — Data Exploration

**AI-Based Early Detection and Classification of Foot and Nail Conditions Using Transfer Learning for Rural Healthcare**

This notebook explores the raw dataset before any preprocessing is written. Its
purpose is to answer the questions that *decide* the preprocessing design:

- How many images are there per class, and how imbalanced is it?
- What resolutions, formats and colour modes are present?
- How many images are corrupt, tiny, or duplicated?

Runs both locally and in Google Colab.

## Setup

The cell below detects Colab, mounts Drive and clones the repo if needed, so the
same notebook works in both places.

**Note:** it clones the `claude/foot-nail-disease-ai-fyrdsb` branch explicitly.
The project code is not on `main` yet, so a plain `git clone` would check out an
empty repository.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# The project code currently lives on this branch, NOT on main. Cloning without
# --branch would check out main, which has no source code in it.
REPO_URL = "https://github.com/gurubasavarajharlapur-jpg/Dissertation_AI_FOOT_NAIL_DISEASE.git"
BRANCH = "claude/foot-nail-disease-ai-fyrdsb"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_ROOT = Path("/content/Dissertation_AI_FOOT_NAIL_DISEASE")
    if PROJECT_ROOT.exists():
        # Already cloned earlier in this runtime: just pull the latest commits.
        !cd {PROJECT_ROOT} && git checkout {BRANCH} && git pull --ff-only
    else:
        !git clone --branch {BRANCH} {REPO_URL} {PROJECT_ROOT}

    %pip install -q -r {PROJECT_ROOT}/requirements.txt

    # Keep the dataset on Drive, not in the runtime: Colab recycles the VM and
    # you do not want to re-download several hundred MB every session.
    DRIVE_DATA = Path("/content/drive/MyDrive/dissertation_foot_nail/data/raw")
    DRIVE_DATA.mkdir(parents=True, exist_ok=True)

    raw_link = PROJECT_ROOT / "data" / "raw"
    if raw_link.is_symlink():
        print(f"data/raw already linked -> {raw_link.resolve()}")
    else:
        # A fresh clone always leaves a .gitkeep here (it is what keeps the
        # otherwise-empty directory in version control), so "contains .gitkeep
        # and nothing else" counts as empty and is safe to replace with the
        # link. Any real file means the dataset is already on the local disk,
        # and silently discarding it would be much worse than skipping the link.
        leftovers = [p for p in raw_link.iterdir() if p.name != ".gitkeep"]
        if leftovers:
            print(
                f"NOTE: {raw_link} contains {len(leftovers)} file(s) already, so it "
                f"was NOT replaced with a\n      link to Drive. The dataset will be "
                f"lost when this runtime is recycled.\n      Move those files to "
                f"{DRIVE_DATA} and re-run this cell to fix that."
            )
        else:
            for p in raw_link.iterdir():
                p.unlink()
            raw_link.rmdir()
            raw_link.symlink_to(DRIVE_DATA)
            print(f"data/raw -> {DRIVE_DATA}")
else:
    # Local: the notebook lives in notebooks/, so the project root is one level up.
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("Project root    :", PROJECT_ROOT)
print("Running in Colab:", IN_COLAB)

In [ ]:
from src import config

print("Target classes :", config.CLASS_NAMES)
print("Input shape    :", config.INPUT_SHAPE)
print("Random seed    :", config.RANDOM_SEED)
print("Splits         : %.0f/%.0f/%.0f train/val/test"
      % (config.TRAIN_SPLIT * 100, config.VAL_SPLIT * 100, config.TEST_SPLIT * 100))

## 1. Fetch the datasets

Two public sources:

| Source | Contents | Link |
|---|---|---|
| Mendeley Data `hsj38fwnvr` v3 | Foot imagery (wound / ulcer) | https://data.mendeley.com/datasets/hsj38fwnvr/3 |
| Figshare `5398573` | Onychomycosis — nail fungal infection | https://figshare.com/articles/dataset/5398573 |

`--list` shows what is available without downloading anything. If your network
blocks these hosts, download them manually in a browser and unzip into
`data/raw/`.

In [ ]:
!python src/download_data.py --list

In [ ]:
# Downloads several hundred MB into data/raw/ (i.e. onto Drive).
# Safe to re-run: files already downloaded are skipped, so an
# interrupted download resumes rather than starting over.
!python src/download_data.py

## 2. Inspect what is in `data/raw/`

This is the key step. Read the output carefully — the folder structure it prints
is what determines how the source folders map onto the four project classes.

In [ ]:
!python src/inspect_data.py

## 3. Dig into the inventory

`inspect_data.py` saves a per-image table. Load it here to explore further.

In [ ]:
import pandas as pd

csv_path = config.METRICS_DIR / "raw_data_files.csv"

# inspect_data.py writes this only on a successful run. If it exited early
# (usually "no images found"), the file is absent and the traceback below is
# far more useful than a bare FileNotFoundError.
if not csv_path.exists():
    raise FileNotFoundError(
        f"{csv_path} does not exist.\n"
        f"src/inspect_data.py writes it only after a successful run — run the "
        f"download and inspection cells above first, and check that neither "
        f"reported an error."
    )

files = pd.read_csv(csv_path)
print(f"{len(files)} images catalogued")
files.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

counts = files["label"].value_counts()
fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(counts))))
sns.barplot(x=counts.to_numpy(), y=counts.index.astype(str), ax=ax, color="#4C72B0")
ax.set(xlabel="Number of images", ylabel="Source folder",
       title="Images per source folder in data/raw/")
for i, v in enumerate(counts.to_numpy()):
    ax.text(v, i, f" {v}", va="center")
plt.tight_layout()
plt.show()

In [ ]:
# Resolution spread: are we mostly downscaling (safe) or upscaling (blurring)?
valid = files[~files["corrupt"]]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(valid["width"], valid["height"], alpha=0.35, s=14)
axes[0].axvline(config.IMAGE_SIZE[0], color="crimson", ls="--", label="target 224")
axes[0].axhline(config.IMAGE_SIZE[1], color="crimson", ls="--")
axes[0].set(xlabel="Width (px)", ylabel="Height (px)", title="Image dimensions")
axes[0].legend()

sns.histplot(valid["aspect_ratio"], bins=40, ax=axes[1], color="#55A868")
axes[1].axvline(1.0, color="crimson", ls="--", label="square")
axes[1].set(xlabel="Aspect ratio (w/h)", title="Aspect ratio distribution")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# A visual sample from each source folder — always look at the actual images
# before trusting any statistic about them.
import matplotlib.image as mpimg

folders = sorted(valid["label"].unique())
per_folder = 4
fig, axes = plt.subplots(len(folders), per_folder,
                         figsize=(3 * per_folder, 3 * len(folders)),
                         squeeze=False)
for row, folder in enumerate(folders):
    subset = valid[valid["label"] == folder].sample(
        min(per_folder, (valid["label"] == folder).sum()),
        random_state=config.RANDOM_SEED,
    )
    for col in range(per_folder):
        ax = axes[row][col]
        ax.axis("off")
        if col < len(subset):
            ax.imshow(mpimg.imread(subset.iloc[col]["path"]))
            if col == 0:
                ax.set_title(folder, loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## 4. Decide the class mapping

Using everything above, fill in how each source folder maps onto the four
project classes. This mapping is the hand-off to `src/preprocessing.py`.

Record the reasoning here too — the viva will ask why each folder was assigned
the way it was, and why any folder was excluded.

In [ ]:
# Fill this in after reading the inspection report.
# Key   = folder path as it appears in the `label` column above
# Value = one of config.CLASS_NAMES, or None to exclude the folder entirely
FOLDER_TO_CLASS = {
    # "figshare_nail/train/onychomycosis": "nail_fungal",
    # "figshare_nail/train/normal":        "healthy",
    # "mendeley_foot/ulcer":               "foot_ulcer",
    # "mendeley_foot/wound":               "foot_wound",
}

for folder in sorted(valid["label"].unique()):
    status = FOLDER_TO_CLASS.get(folder, "UNMAPPED")
    print(f"  {folder:<50} -> {status}")

## Findings

_Write the conclusions here once the cells above have been run:_

- **Total usable images:**
- **Class balance:** (imbalance ratio, and whether class weighting is needed)
- **Resolution:** (what fraction would be upscaled to 224x224)
- **Formats/modes:** (any non-RGB conversions required)
- **Quality issues:** (corrupt, tiny, duplicate counts)
- **Decisions for preprocessing:**